In [ ]:
import os
import re
from collections import Counter
from itertools import chain

import numpy as np
import pandas as pd

from utils.utils import clean, clean_one_speaker_tag, clean_text, get_df, join_dfs
from utils.viz import visualize_annotations

pd.set_option("display.max_colwidth", None)

In [ ]:
TEXT = "01_Text"
text_corpus = {
    "m": TEXT + "/First_group/1_M.csv",
    "c": TEXT + "/First_group/2_C.csv",
    "g": TEXT + "/First_group/3_G.csv",
    "a": TEXT + "/Second_group/1_A.csv",
    "r": TEXT + "/Second_group/2_R.csv",
    "n": TEXT + "/Second_group/3_N.csv",
}

In [ ]:
# Text
dfs = {k: get_df(file) for k, file in text_corpus.items()}

In [ ]:
# TEXTS agregats
joined = pd.read_csv("01_Text/text_with_mmlabel.csv")
joined["len"] = joined.text_clean.str.len()

# Filtre mateixa anotació text i multimodal (vídeo) + agreement anotadors
joined = joined[(joined.sexist_text == joined.sexist_video) & (joined.sexist_soft.isin((0, 1)))].sort_values(by="len")
IDS = joined.id.tolist()

In [ ]:
def get_lens(_df, person):
    df = _df.copy()
    df["len"] = df.text.str.len()
    df = df[(df.sentiment == "SEXIST") & (df.video_id.isin(IDS)) & (df.len < 2000)]
    # df = df[(df.video_id.isin(IDS))].sort_values(by="len")

    df["label"] = df["label"].apply(lambda x: eval(x) if isinstance(x, str) else (x if isinstance(x, list) else []))
    df[["text_clean", "label_clean"]] = df.apply(clean, axis=1)
    df["span_len"] = df.label_clean.apply(lambda x: sum([ann["end"] - ann["start"] for ann in x]))
    df["span_num"] = df.label_clean.str.len()
    df["span_lens"] = df.label_clean.apply(lambda x: [ann["end"] - ann["start"] for ann in x])
    return df

In [ ]:
for k, df in dfs.items():
    print(k)
    dfs[k] = get_lens(df, k)

In [ ]:
row = dfs["c"].iloc[0]
row.label_clean

In [ ]:
ID_COL = "video_id"
binary_cols = [
    "sexist",
    # "irony",
    # "humor",
    # "implicit sexism",
    # "stereotypes",
    # "inequality",
    # "discrimination",
    # "objectification",
    # "critique",
    # "reported_sexism",
    # "sexist_irony",
    # "sexist_humor",
    # "no_sexist_irony",
    # "no_sexist_humor",
]
num_cols = [
    "span_len",
    "span_num",
]
label_col = "label_clean"
cols = binary_cols + num_cols + [label_col]

cols_left = [ID_COL, "text_clean"] + cols
cols_right = [ID_COL] + cols

df = join_dfs(dfs, cols_left, cols_right, ID_COL, binary_cols, num_cols, label_col)
# df["text_clean"] = df.text.apply(clean_text).apply(clean_one_speaker_tag)
df["len"] = df.text_clean.str.len()
df = df.sort_values(by=["span_len", "len"])
# df[["id", "text_clean"] + binary_cols + [c + "_soft" for c in binary_cols] + num_cols].to_csv(
#     os.path.join(TEXT, "text.csv"), index=False
# )

In [ ]:
row = df.iloc[0]
for ann in row["label_clean"]:
    print("---", ann["text"])

In [ ]:
row.label_clean

In [ ]:
df.len.hist(), df.shape

In [ ]:
# df.sort_values(by=["span_len", "len"])

In [ ]:
row.label_clean

In [ ]:
print(row.text_clean)

In [ ]:
import re

re.sub(r"a([0-9])", r"b\1", "a2, a3, a4")

In [ ]:
row.text_clean

In [ ]:
for i in range(3):
    print("------------")
    row = df.iloc[i]
    anotacions = row.label_clean
    visualize_annotations(row.text_clean, row.label_clean, file=f"{i}.pdf")

In [ ]:
Counter(chain.from_iterable([anot["labels"] for anot in anotacions]))